# 03 - Model Training
## Loan Risk Assessment System

**Objective:** Split the engineered dataset, scale features, and train three classification models: Logistic Regression, Decision Tree (GridSearchCV-tuned), and Random Forest (GridSearchCV-tuned).

**Workflow:**
1. Load engineered dataset
2. Train/test split (80/20, stratified)
3. Feature scaling (Logistic Regression only)
4. Train Logistic Regression
5. Tune & train Decision Tree
6. Tune & train Random Forest
7. Persist all models

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.model_selection import train_test_split
from src import config
from src.data_loader import load_dataset
from src.feature_engineering import engineer_features
from src.model import scale_features, train_all_models
from src.utils import save_object

cleaned_df = pd.read_csv(config.CLEANED_DATA_PATH)
engineered_df = engineer_features(cleaned_df)
engineered_df.shape

[2026-07-31 19:26:27] INFO - src.feature_engineering - Starting feature engineering pipeline...


[2026-07-31 19:26:27] INFO - src.feature_engineering - Created feature: 'total_income'


[2026-07-31 19:26:27] INFO - src.feature_engineering - Created feature: 'loan_income_ratio'


[2026-07-31 19:26:27] INFO - src.feature_engineering - Converted 'Dependents' to numeric (3+ mapped to 3)


[2026-07-31 19:26:27] INFO - src.feature_engineering - Encoded target column 'Loan_Status': Y -> 1, N -> 0


[2026-07-31 19:26:27] INFO - src.feature_engineering - One-hot encoded columns: ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']. New shape: (614, 15)


[2026-07-31 19:26:27] INFO - src.feature_engineering - Feature engineering complete. Final shape: (614, 15)


(614, 15)

## Step 1: Train/Test Split (80/20, stratified, random_state=42)

In [2]:
X = engineered_df.drop(columns=[config.TARGET_COLUMN])
y = engineered_df[config.TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Train approval rate: {y_train.mean():.3f}')
print(f'Test approval rate: {y_test.mean():.3f}')

Train shape: (491, 14)
Test shape: (123, 14)
Train approval rate: 0.745
Test approval rate: 0.748


## Step 2: Feature Scaling

`StandardScaler` is fit on the training set and applied to both train/test — used exclusively for Logistic Regression since tree-based models (Decision Tree, Random Forest) are scale-invariant.

In [3]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
save_object(scaler, config.MODELS_DIR / config.MODEL_FILENAMES['scaler'])

[2026-07-31 19:26:27] INFO - src.model - Fitted StandardScaler on training data and transformed train/test sets.


[2026-07-31 19:26:27] INFO - loan_risk_assessment - Saved object to: /home/claude/AIML-BonusProject-Loan-Risk-Assessment/models/scaler.pkl


## Step 3: Train All Models

- **Logistic Regression**: trained on scaled features with `class_weight='balanced'`.
- **Decision Tree**: tuned via `GridSearchCV` over `max_depth`, `min_samples_split`, `min_samples_leaf` (cv=5, scoring=roc_auc).
- **Random Forest**: tuned via `GridSearchCV` over `n_estimators`, `max_depth`, `min_samples_split`, `max_features` (cv=5, scoring=roc_auc).

In [4]:
models = train_all_models(
    X_train, y_train, X_train_scaled,
    dt_param_grid=config.DECISION_TREE_PARAM_GRID,
    rf_param_grid=config.RANDOM_FOREST_PARAM_GRID,
    cv=config.CV_FOLDS, scoring=config.SCORING_METRIC,
    random_state=config.RANDOM_STATE
)
models


MODEL TRAINING
[2026-07-31 19:26:27] INFO - src.model - Training Logistic Regression...


[2026-07-31 19:26:27] INFO - src.model - Trained Logistic Regression model.


[2026-07-31 19:26:27] INFO - src.model - Training Decision Tree (with GridSearchCV tuning)...


[2026-07-31 19:26:30] INFO - src.model - Decision Tree GridSearchCV best params: {'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 2}


[2026-07-31 19:26:30] INFO - src.model - Decision Tree GridSearchCV best CV roc_auc: 0.6959


[2026-07-31 19:26:30] INFO - src.model - Training Random Forest (with GridSearchCV tuning)...


[2026-07-31 19:28:13] INFO - src.model - Random Forest GridSearchCV best params: {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}


[2026-07-31 19:28:13] INFO - src.model - Random Forest GridSearchCV best CV roc_auc: 0.6819


{'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=3,
                        min_samples_leaf=10, random_state=42),
 'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=4, n_jobs=-1,
                        random_state=42)}

In [5]:
print('Decision Tree best params:', models['Decision Tree'].get_params())

Decision Tree best params: {'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 10, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}


In [6]:
print('Random Forest best params:', models['Random Forest'].get_params())

Random Forest best params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 4, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## Step 4: Persist Trained Models

In [7]:
save_object(models['Logistic Regression'], config.MODELS_DIR / config.MODEL_FILENAMES['logistic_regression'])
save_object(models['Decision Tree'], config.MODELS_DIR / config.MODEL_FILENAMES['decision_tree'])
save_object(models['Random Forest'], config.MODELS_DIR / config.MODEL_FILENAMES['random_forest'])

[2026-07-31 19:28:13] INFO - loan_risk_assessment - Saved object to: /home/claude/AIML-BonusProject-Loan-Risk-Assessment/models/logistic_regression.pkl


[2026-07-31 19:28:13] INFO - loan_risk_assessment - Saved object to: /home/claude/AIML-BonusProject-Loan-Risk-Assessment/models/decision_tree.pkl


[2026-07-31 19:28:13] INFO - loan_risk_assessment - Saved object to: /home/claude/AIML-BonusProject-Loan-Risk-Assessment/models/random_forest.pkl


## Conclusion

All three models were successfully trained:
- Logistic Regression was trained on standardized features with balanced class weights to account for the class imbalance toward approvals.
- The Decision Tree and Random Forest were tuned via 5-fold cross-validated grid search, optimizing for ROC-AUC.
- All trained models and the fitted scaler were persisted to `models/` for reuse in evaluation (`04_Model_Evaluation.ipynb`) without retraining.